首先需要安装 selenium。  
复制执行 `pip install selenium`，可按如下方式：
1. 打开 PowerShell，然后执行 `python pip install selenium`
2. 或者安装到本项目级别，只需把 `pip` 加到本项目：
   1. 打开新终端（VS Code 或 Cursor）
   2. 执行 `python -m ensurepip --upgrade`
   3. 再运行 `python pip install selenium`


下面用 Python 读取站点，等待页面完全渲染后，把结果以 HTML 字符串返回。

In [ ]:

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

def readClientRenderedHTML(url):
    # Chrome 启动选项
    options = Options()
    #options.add_argument("--headless")  # 无界面运行浏览器

    # 启动浏览器
    driver = webdriver.Chrome(options=options)

    # 打开页面
    driver.get(url)

    # 等待 JavaScript 渲染
    time.sleep(3)

    # 获取渲染后的 HTML
    html = driver.page_source


    # 关闭浏览器
    driver.quit()

    return html

现在测试新函数，直接运行下面的代码即可。

In [ ]:
# 测试：抓取个人站点渲染后的 HTML
result  = readClientRenderedHTML("https://khaldoon-saqallah.com")
print(result)

接下来用该函数结合 AI，像第 1 周第 1 天那样，询问页面内容。

In [ ]:
# 用 Selenium 抓取页面 + OpenAI 生成俏皮摘要
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

openai = OpenAI(api_key=api_key)

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + "\n\n" + website}
    ]


def summarize(url):
    website = readClientRenderedHTML(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

 

现在执行（调用）上面定义的 `display_summary` 函数。

In [ ]:
# 对目标 URL 展示摘要
display_summary("https://khaldoon-saqallah.com")